# Intro 3: TabularX Regression -- Diabetes Dataset

This tutorial demonstrates TabularX on a regression task.


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
from sklearn.datasets import load_diabetes
import tabularx as tbx
print("TabularX ready")


In [ ]:
data = load_diabetes(as_frame=True)
X, y = data.data, data.target
print(f"Shape: {X.shape}\nTarget range: {y.min():.1f} ~ {y.max():.1f} (mean={y.mean():.1f})")


## Create AutoEstimator

Use `task='regression'`. Default metric is RMSE (lower is better).


In [ ]:
ae = tbx.AutoEstimator(task="regression", output_dir="./intro3-output", pdf_report=True)
ae.param.model_search.set_param("n_trials", 30)
ae.param.model_search.set_param("top_k_models", 2)
ae.param.shap.set_param("top_k_features", 10)
ae.param.split.set_param("test_size", 0.2)
ae.param.split.set_param("validation_size", 0.25)
print(ae.param.overview())


## Run pipeline


In [ ]:
ae.fit(X, y)
print("Done!")


## Results


In [ ]:
result = ae.result
print(f"Task: {result.searched.task_type}, Metric: {result.searched.primary_metric}")
for i, r in enumerate(result.searched.ranking, 1):
    v = r.validation_metrics.get("rmse", 0)
    t = (r.test_metrics or {}).get("rmse", 0)
    print(f"{i}. {r.model_name:>20} valid_rmse={v:.4f} test_rmse={t:.4f}")
print(f"\nBest: {result.searched.best_model_name}")


## SHAP


In [ ]:
shap = result.shap
if shap:
    print(f"Model: {shap.model_name}, Samples: {shap.n_explain_samples}")
    print(f"Top features: {shap.top_k_features[:5]}")
else:
    print("No SHAP")


## Summary

Regression uses `task='regression'`, defaults to RMSE metric, and generates no confusion matrix or ROC curve. SHAP still works.


In [ ]:
from pathlib import Path
print("Generated files:")
for name, p in result.report_paths.items():
    ok = "OK" if Path(p).exists() else "MISS"
    print(f"  [{ok}] {name}: {p}")
